# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



### a) Create Agent

In [1]:
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()

# Create an artificial dialogue history for testing
from gai.messages import FileDialogue, MessagePydantic
messages = [
    MessagePydantic(**{
        'id': 'b1e5f98c-f6eb-47de-a6e2-387510d970f9', 
        'header': {
            'sender': 'User',
            'recipient': 'Sara',
            "timestamp": 1751308157.270983,
            "order": 0            
        }, 'body': {
            'type': 'chat.send',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 0,
            'role': "user",
            'content': 'It is a very nice weather in Singapore right now.',
        }
    }),
    MessagePydantic(**{
        'id': 'abbc7961-45dc-4973-aaf4-a6224ed35d37', 
        'header': {
            'sender': 'Sara',
            'recipient': 'User',
            "timestamp": 1751308167.3488164,
            "order": 1
        }, 'body': {
            'type': 'chat.reply',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0, 
            'step_no': 1,
            'chunk_no':10,
            'chunk':'<eom>',
            'role': "assistant",
            'content': 'Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It\'s a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?'
        }
    })]
from gai.lib.constants import DEFAULT_GUID
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages,file_path=file_path)
recap = dialogue.extract_recap()

agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
    recap=recap
)

### reset monologue (optional)

In [2]:
monologue.reset()


### b) start

The location of the public holiday is inferred from the dialogue context.

In [3]:
goal = "When is the next public holiday?"
user_message=f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
"""

resp=agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

# Update dialogue
dialogue.add_user_message(recipient="Sara", content=goal)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

I'll help you find the next public holiday. To provide accurate information, I need to know your location since public holidays vary by country and region. Let me first get the current date and then search for upcoming public holidays.
Now I need to know your location to find the relevant public holidays. Could you please tell me which country or region you're in? This will help me search for the most accurate information about upcoming public holidays in your area.


MessagePydantic(id='d58d6c2a-e161-402a-9477-abf887330e3c', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1752652642.7281542, order=1), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=0, step_no=1, message_id='00000000-0000-0000-0000-000000000000.33', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content="Now I need to know your location to find the relevant public holidays. Could you please tell me which country or region you're in? This will help me search for the most accurate information about upcoming public holidays in your area."))

### c) Show monologue

In [4]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "812ba9e0-4fc0-4510-bdd2-fe0964a8eb9a",
    "header": {
        "sender": "User",
        "recipient": "ToolUseAgent",
        "timestamp": 1752652448.0490608,
        "order": 2
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 4,
        "content_type": "text",
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_01X2jUEpRpkdq6vn7eYwtjtL",
                "content": "Current UTC time is 2025-07-16, and the time in UTC is 2025-07-16."
            }
        ]
    }
}
{
    "id": "9651b2a3-547b-4b38-8545-43b8f637c97f",
    "header": {
        "sender": "ToolUseAgent",
        "recipient": "User",
        "timestamp": 1752652454.052774,
        "order": 3
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 4,
        "conten

### d) resume

In [4]:
resp = agent.resume()
# Stream the response
can_resume=False
async for chunk in resp:
    can_resume=True
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
if not can_resume:
    print("Cannot resume because user has not provided input.")


Cannot resume because user has not provided input.


### e) User interrupt agent

Should be able to interrupt the agent and continue with original task.

In [5]:
user_message = "Tell me a one paragraph joke."
resp = agent.interrupt(user_message=user_message)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

# Update dialogue when user interrupt agent
dialogue.add_user_message(recipient="Sara", content=user_message)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

Here's a one-paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, but there's nothing there. The librarian chuckles and points to the shelf directly behind him, saying, "No, the books on paranoia are literally right behind you on that shelf." The man sighs in relief, grabs a book, and heads to checkout. As he's leaving, the librarian calls out, "By the way, that book is overdue!" The man drops everything and runs out screaming, "I knew they were watching me!"

Now, let's get back on track with finding your next public holiday! To provide you with accurate information about upcoming public holidays, I still need to know which country or region you're located in. Different countries have different public holidays and dates, so your location will help me search for the most relevant information for you.


MessagePydantic(id='6a5e6e19-7fbe-4d03-b564-1e23722e0ba3', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1752652676.635989, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.95', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content='Here\'s a one-paragraph joke for you:\n\nA man walks into a library and asks for books on paranoia. The librarian whispers, "They\'re right behind you!" The man spins around frantically, but there\'s nothing there. The librarian chuckles and points to the shelf directly behind him, saying, "No, the books on paranoia are literally right behind you on that shelf." The man sighs in relief, grabs a book, and heads to checkout. As he\'s leaving, the librarian calls out, "By the way, that book is overdue!" The man drops everything and runs out screaming, "I knew they were watching me!"\n\n

### f) Show dialogue

In [6]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: Sara, When is the next public holiday?
Sara: Now I need to know your location to find the relevant public holidays. Could you please tell me which country or region you're in? This will help me search for the most accurate information about upcoming public holidays in your area.
User: Sara, Tell me a one paragraph joke.
Sara: Here's a one-paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, but there's nothing there. The librarian chuckles and points to the shelf directly behind him, saying, "No, the books on paranoia are literally right behind you on that shelf." The man sighs in relief, grabs a book, and heads to checkout. As he's leaving, the librarian calls out, "By the way, that book is overdue!" The man drops everything and runs out screaming, "I knew they were watching me!"

Now, let's get back on track with finding your next public holiday! To provide you w

---
## Scenario 1 - Agent interrupts itself and User has no input

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input

In [7]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [8]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time! First, let me ask you about your timezone preference.
I need to know your timezone to provide you with the accurate current time. Could you please tell me your timezone? You can provide it in IANA format (like "America/New_York", "Europe/London", "Asia/Tokyo", etc.) or just tell me your city/country and I'll help determine the correct timezone.


Confirm that "resume" returns nothing because the agent is waiting for user input.

In [9]:
resp = agent.resume()
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
if not can_resume:
    print("\nAgent cannot resume because User did not provide input.")
assert not can_resume, "Agent should not be able to resume while waiting for input."



Agent cannot resume because User did not provide input.


---
## Scenario 2: LLM Interrupts itself and User has input

- LLM Interrupts itself to ask user question
- User responds with answer

In [10]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [11]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time! First, let me ask you about your timezone preference.
What timezone would you like me to use for the current time? Please provide the timezone in IANA format (e.g., "America/New_York", "Europe/London", "Asia/Shanghai", "UTC", etc.). If you're not sure, you can also tell me your city or region and I'll help determine the appropriate timezone.


Confirm user cannot continue because LLM is waiting for user input.

In [12]:
resp = agent.resume()
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
if not can_resume:
    print("\nAgent cannot resume because User did not provide input.")
assert not can_resume, "Agent should not be able to resume while waiting for input."



Agent cannot resume because User did not provide input.


Confirm that "resume" with message will work when user responds.

In [13]:
resp = agent.resume(user_message="Use SGT")
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
assert can_resume, "Agent should be able to resume after user provides input."


Perfect! I'll get the current time in Singapore Standard Time (SGT) for you.


User can resume normally.

In [14]:
resp = agent.resume()
async for chunk in resp:
    if chunk and isinstance(chunk, str):
        print(chunk, end="", flush=True)

The current time in Singapore Standard Time (SGT) is:

**2025-07-16 16:00:19**

That's 4:00:19 PM on July 16th, 2025, Singapore time.
